In [33]:
from langgraph.graph import StateGraph , START, END
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
from typing import TypedDict, Literal
from dotenv import load_dotenv

load_dotenv()

True

In [34]:
model= ChatGroq(model='llama-3.3-70b-versatile')

In [35]:
class SentimentSchema(BaseModel):
    sentiment: Literal['POSITIVE','NEGATIVE'] = Field(description='Sentiment of the review')

In [49]:
class DiagnosisSchema(BaseModel):
    issueType: Literal['UX','Performance','Bug','Support','Other']= Field(description='the category of the issue mentioned')
    tone: Literal['angry','frustrated','dissappointed','calm']=Field(description='emotional tone of the user')
    urgency: Literal['low','medium','high']=Field(description='type of urgency the issue appears to be')

In [50]:
structured_model=model.with_structured_output(SentimentSchema)
structured_model2=model.with_structured_output(DiagnosisSchema)

In [51]:
class ReviewState(TypedDict):
    review:str
    sentiment: Literal["POSITIVE","NEGATIVE"]
    diagnosis:dict
    response:str

In [59]:
def find_sentiment(state: ReviewState):
    prompt=f'for the following review find out the sentiment\n{state['review']}'
    sentiment=structured_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}

def check_sentiment(state:ReviewState)-> Literal["positive_response","run_diagnosis"]:
    if state['sentiment']=='POSITIVE':
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state: ReviewState):
    prompt=f""" write a warm thank you message in response to this review:\n\n {state['review']}\n Also kindly ask the user to leave feedback on our website."""
    response= model.invoke(prompt).content
    return {'response':response}
def run_diagnosis(state:ReviewState):
    prompt=f""" Diagnose this negative review:\n\n{state['review']}\n "Return issueType,tone and urgency"."""
    response= structured_model2.invoke(prompt)
    return {'diagnosis':response.model_dump()}
def negative_response(state:ReviewState):
    diagnosis=state['diagnosis']
    prompt=f""" You are a support assistant. The user had a '{diagnosis['issueType']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}' . Write and empatehtic and helpful resolution message"""
    response = model.invoke(prompt).content
    return {'response':response}

In [60]:
graph=StateGraph(ReviewState)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)
graph.add_node('run_diagnosis',run_diagnosis)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)
workflow=graph.compile()


In [61]:
initial_state={
    'review':"the product usage was bad and the price is too hight"
}
final_state=workflow.invoke(initial_state)
print(final_state)

{'review': 'the product usage was bad and the price is too hight', 'sentiment': 'NEGATIVE', 'diagnosis': {'issueType': 'Support', 'tone': 'angry', 'urgency': 'high'}, 'response': "Dear [User],\n\nI want to start by apologizing for the inconvenience you're experiencing with your issue. I can sense the frustration and urgency in your message, and I'm here to help resolve the problem as quickly and efficiently as possible.\n\nI understand how frustrating it can be when things don't go as planned, and I appreciate you reaching out to us for support. I'm committed to providing you with a helpful and empathetic resolution.\n\nTo better assist you, I'd like to request some additional information about the issue you're facing. Could you please provide me with more details about the problem, including any error messages or symptoms you've encountered? This will help me to better understand the issue and provide a more accurate solution.\n\nIn the meantime, I want to assure you that I'm prioriti